In [ ]:
""" 
This file was built to solve numerically a classical PDE, 1D wave equation. The equation corresponds to :
$\dfrac{\partial}{\partial x} \left( \dfrac{\partial c^2 U}{\partial x} \right) = \dfrac{\partial^2 U}{\partial t^2}$
 
where
 - U represent the signal
 - x represent the position
 - t represent the time
 - c represent the velocity of the wave (depends on space parameters)

The numerical scheme is based on finite difference method. This program is also providing several boundary conditions. More particularly the Neumann, Dirichlet and Mur boundary conditions.
Copyright - © SACHA BINDER - 2021
"""

import matplotlib.pyplot as plt

############## MODULES IMPORTATION ###############
import numpy as np
import viz_tools  # self-developed module that groups animation functions


# Def of the initial condition
def I(x):
    """
    Single space variable fonction that
    represent the wave form at t = 0
    """
    # return np.sin(x)+0.5
    return np.exp(-((x - 0.5) ** 2) / 0.01)


############## SET-UP THE PROBLEM ###############
# Def of velocity (spatial scalar field)
def celer(x):
    """
    constant velocity
    """
    return 1


left_bound_cond = 2  # Boundary cond 1 : Dirichlet, 2 : Neumann, 3 Mur
right_bound_cond = 2  # Boundary cond 1 : Dirichlet, 2 : Neumann, 3 Mur

# Spatial mesh - i indices
L_x = 1.5  # Range of the domain according to x [m]
dx = 0.01  # Infinitesimal distance
N_x = int(L_x / dx)  # Points number of the spatial mesh
X = np.linspace(0, L_x, N_x + 1)  # Spatial array

# Temporal mesh with CFL < 1 - j indices
L_t = 4  # Duration of simulation [s]
dt = 0.01 * dx  # Infinitesimal time with CFL (Courant–Friedrichs–Lewy condition)
N_t = int(L_t / dt)  # Points number of the temporal mesh
T = np.linspace(0, L_t, N_t + 1)  # Temporal array

# Velocity array for calculation (finite elements)
c = np.zeros(N_x + 1, float)
for i in range(0, N_x + 1):
    c[i] = celer(X[i])

############## CALCULATION CONSTANTS ###############
c_1 = c[0]
c_2 = c[N_x]

C2 = (dt / dx) ** 2

CFL_1 = c_1 * (dt / dx)
CFL_2 = c_2 * (dt / dx)

############## PROCESSING LOOP ###############

# $\forall i \in {0,...,N_x}$
u_jm1 = np.zeros(N_x + 1, float)  # Vector array u_i^{j-1}
u_j = np.zeros(N_x + 1, float)  # Vector array u_i^j
u_jp1 = np.zeros(N_x + 1, float)  # Vector array u_i^{j+1}

q = np.zeros(N_x + 1, float)
q[0 : N_x + 1] = c[0 : N_x + 1] ** 2

U = np.zeros((N_x + 1, N_t + 1), float)  # Global solution

# init cond - at t = 0
u_j[0 : N_x + 1] = I(X[0 : N_x + 1])
U[:, 0] = u_j.copy()

# init cond - at t = 1
# without boundary cond
u_jp1[1:N_x] = u_j[1:N_x] + 0.5 * C2 * (
    0.5 * (q[1:N_x] + q[2 : N_x + 1]) * (u_j[2 : N_x + 1] - u_j[1:N_x])
    - 0.5 * (q[0 : N_x - 1] + q[1:N_x]) * (u_j[1:N_x] - u_j[0 : N_x - 1])
)

# left boundary conditions
if left_bound_cond == 1:
    # Dirichlet bound cond
    u_jp1[0] = 0

elif left_bound_cond == 2:
    # Nuemann bound cond
    # i = 0
    u_jp1[0] = u_j[0] + 0.5 * C2 * (
        0.5 * (q[0] + q[0 + 1]) * (u_j[0 + 1] - u_j[0])
        - 0.5 * (q[0] + q[0 + 1]) * (u_j[0] - u_j[0 + 1])
    )

elif left_bound_cond == 3:
    # Mur bound cond
    # i = 0
    u_jp1[0] = u_j[1] + (CFL_1 - 1) / (CFL_1 + 1) * (u_jp1[1] - u_j[0])

# right boundary conditions
if right_bound_cond == 1:
    # Dirichlet bound cond
    u_jp1[N_x] = 0
elif right_bound_cond == 2:
    # Nuemann bound cond
    # i = N_x
    u_jp1[N_x] = u_j[N_x] + 0.5 * C2 * (
        0.5 * (q[N_x - 1] + q[N_x]) * (u_j[N_x - 1] - u_j[N_x])
        - 0.5 * (q[N_x - 1] + q[N_x]) * (u_j[N_x] - u_j[i - 1])
    )
elif right_bound_cond == 3:
    # Mur bound cond
    # i = N_x
    u_jp1[N_x] = u_j[N_x - 1] + (CFL_2 - 1) / (CFL_2 + 1) * (u_jp1[N_x - 1] - u_j[N_x])

u_jm1 = u_j.copy()  # go to the next step
u_j = u_jp1.copy()  # go to the next step
U[:, 1] = u_j.copy()


# Process loop (on time mesh)
for j in range(1, N_t):
    # calculation at step j+1
    # without boundary cond
    u_jp1[1:N_x] = (
        -u_jm1[1:N_x]
        + 2 * u_j[1:N_x]
        + C2
        * (
            0.5 * (q[1:N_x] + q[2 : N_x + 1]) * (u_j[2 : N_x + 1] - u_j[1:N_x])
            - 0.5 * (q[0 : N_x - 1] + q[1:N_x]) * (u_j[1:N_x] - u_j[0 : N_x - 1])
        )
    )

    # left bound conditions
    if left_bound_cond == 1:
        # Dirichlet bound cond
        u_jp1[0] = 0

    elif left_bound_cond == 2:
        # Nuemann bound cond
        # i = 0
        u_jp1[0] = (
            -u_jm1[0]
            + 2 * u_j[0]
            + C2
            * (
                0.5 * (q[0] + q[0 + 1]) * (u_j[0 + 1] - u_j[0])
                - 0.5 * (q[0] + q[0 + 1]) * (u_j[0] - u_j[0 + 1])
            )
        )

    elif left_bound_cond == 3:
        # Mur bound cond
        # i = 0
        u_jp1[0] = u_j[1] + (CFL_1 - 1) / (CFL_1 + 1) * (u_jp1[1] - u_j[0])

    # right bound conditions
    if right_bound_cond == 1:
        # Dirichlet bound cond
        u_jp1[N_x] = 0

    elif right_bound_cond == 2:
        # Nuemann bound cond
        # i = N_x
        u_jp1[N_x] = (
            -u_jm1[N_x]
            + 2 * u_j[N_x]
            + C2
            * (
                0.5 * (q[N_x - 1] + q[N_x]) * (u_j[N_x - 1] - u_j[N_x])
                - 0.5 * (q[N_x - 1] + q[N_x]) * (u_j[N_x] - u_j[N_x - 1])
            )
        )

    elif right_bound_cond == 3:
        # Mur bound cond
        # i = N_x
        u_jp1[N_x] = u_j[N_x - 1] + (CFL_2 - 1) / (CFL_2 + 1) * (u_jp1[N_x - 1] - u_j[N_x])

    u_jm1[:] = u_j.copy()  # go to the next step
    u_j[:] = u_jp1.copy()  # go to the next step
    U[:, j] = u_j.copy()

############## PLOT ###############

anim1 = viz_tools.anim_1D(X, U, dt, 100, save=True, myxlim=(0, 1.5), myylim=(-0.5, 1.5))

In [ ]:
import numpy as np
import torch
from tqdm.notebook import tqdm


def _generate_gaussian(num, seq_length, dim, gamma=0.1):
    """_summary_

    Args:
        num (_type_): _description_
        seq_length (_type_): _description_
        dim (_type_): _description_

    Returns:
        _type_: _description_
    """
    import numpy as np
    from sklearn.metrics.pairwise import rbf_kernel

    xs = np.arange(seq_length) * 0.1
    mean = [0 for _ in xs]
    gram = rbf_kernel(xs[:, np.newaxis], gamma=gamma)

    ys = np.random.multivariate_normal(mean, gram, size=(num, dim))
    return ys.transpose(0, 2, 1)


def solve_wave_equation(data_num, celer, left_bound_cond, right_bound_cond, L_x, dx, L_t, dt):
    """
    Solve the 1D wave equation numerically using finite difference method.

    Parameters:
        I: function
            Initial condition function of space.
        celer: function
            Function defining the velocity field.
        left_bound_cond: int
            Type of boundary condition on the left boundary (1: Dirichlet, 2: Neumann, 3: Mur).
        right_bound_cond: int
            Type of boundary condition on the right boundary (1: Dirichlet, 2: Neumann, 3: Mur).
        L_x: float
            Range of the domain according to x.
        dx: float
            Infinitesimal distance for spatial mesh.
        L_t: float
            Duration of simulation.
        dt: float
            Infinitesimal time.

    Returns:
        U: numpy.ndarray
            Array containing the solution of the wave equation.
    """
    # Spatial mesh
    N_x = int(L_x / dx)
    X = torch.linspace(0, L_x, N_x + 1)

    # Temporal mesh
    N_t = int(L_t / dt)
    # T = np.linspace(0, L_t, N_t + 1)

    # Velocity array for calculation
    c = celer(X)
    q = c**2

    # Calculation constants

    c_1 = c[0]
    c_2 = c[N_x]

    C2 = (dt / dx) ** 2

    CFL_1 = c_1 * (dt / dx)
    CFL_2 = c_2 * (dt / dx)

    # Initialize solution array
    U = torch.zeros(data_num, N_x + 1, N_t + 1)

    # Initialize u_jm1, u_j, u_jp1
    u_jm1 = torch.zeros(data_num, N_x + 1)
    u_j = torch.zeros(data_num, N_x + 1)
    u_jp1 = torch.zeros(data_num, N_x + 1)

    # Initial conditions
    I = torch.tensor(_generate_gaussian(data_num, N_x + 1, 1, gamma=0.1)[:, :, 0])

    # Initial condition at t = 0
    u_j[:, 0 : N_x + 1] = I
    U[:, :, 0] = u_j.clone()

    u_jp1[:, 1:N_x] = u_j[:, 1:N_x] + 0.5 * C2 * (
        0.5 * (q[1:N_x] + q[2 : N_x + 1]) * (u_j[:, 2 : N_x + 1] - u_j[:, 1:N_x])
        - 0.5 * (q[0 : N_x - 1] + q[1:N_x]) * (u_j[:, 1:N_x] - u_j[:, 0 : N_x - 1])
    )

    # left boundary conditions
    if left_bound_cond == 1:
        # Dirichlet bound cond
        u_jp1[:, 0] = 0

    elif left_bound_cond == 2:
        # Nuemann bound cond
        # i = 0
        u_jp1[:, 0] = u_j[:, 0] + 0.5 * C2 * (
            0.5 * (q[0] + q[0 + 1]) * (u_j[:, 0 + 1] - u_j[:, 0])
            - 0.5 * (q[0] + q[0 + 1]) * (u_j[:, 0] - u_j[:, 0 + 1])
        )

    elif left_bound_cond == 3:
        # Mur bound cond
        # i = 0
        u_jp1[:, 0] = u_j[:, 1] + (CFL_1 - 1) / (CFL_1 + 1) * (u_jp1[:, 1] - u_j[:, 0])

    # right boundary conditions
    if right_bound_cond == 1:
        # Dirichlet bound cond
        u_jp1[:, N_x] = 0
    elif right_bound_cond == 2:
        # Nuemann bound cond
        # i = N_x
        u_jp1[:, N_x] = u_j[:, N_x] + 0.5 * C2 * (
            0.5 * (q[N_x - 1] + q[N_x]) * (u_j[:, N_x - 1] - u_j[:, N_x])
            - 0.5 * (q[N_x - 1] + q[N_x]) * (u_j[:, N_x] - u_j[:, N_x - 1])
        )
    elif right_bound_cond == 3:
        # Mur bound cond
        # i = N_x
        u_jp1[:, N_x] = u_j[:, N_x - 1] + (CFL_2 - 1) / (CFL_2 + 1) * (
            u_jp1[:, N_x - 1] - u_j[:, N_x]
        )

    u_jm1 = u_j.clone()  # go to the next step
    u_j = u_jp1.clone()  # go to the next step
    U[:, :, 1] = u_j.clone()

    for j in tqdm(range(1, N_t)):
        # calculation at step j+1
        # without boundary cond
        u_jp1[:, 1:N_x] = (
            -u_jm1[:, 1:N_x]
            + 2 * u_j[:, 1:N_x]
            + C2
            * (
                0.5 * (q[1:N_x] + q[2 : N_x + 1]) * (u_j[:, 2 : N_x + 1] - u_j[:, 1:N_x])
                - 0.5 * (q[0 : N_x - 1] + q[1:N_x]) * (u_j[:, 1:N_x] - u_j[:, 0 : N_x - 1])
            )
        )

        # left bound conditions
        if left_bound_cond == 1:
            # Dirichlet bound cond
            u_jp1[:, 0] = 0

        elif left_bound_cond == 2:
            # Nuemann bound cond
            # i = 0
            u_jp1[:, 0] = (
                -u_jm1[:, 0]
                + 2 * u_j[:, 0]
                + C2
                * (
                    0.5 * (q[0] + q[0 + 1]) * (u_j[:, 0 + 1] - u_j[:, 0])
                    - 0.5 * (q[0] + q[0 + 1]) * (u_j[:, 0] - u_j[:, 0 + 1])
                )
            )

        elif left_bound_cond == 3:
            # Mur bound cond
            # i = 0
            u_jp1[:, 0] = u_j[:, 1] + (CFL_1 - 1) / (CFL_1 + 1) * (u_jp1[:, 1] - u_j[:, 0])

        # right bound conditions
        if right_bound_cond == 1:
            # Dirichlet bound cond
            u_jp1[:, N_x] = 0

        elif right_bound_cond == 2:
            # Nuemann bound cond
            # i = N_x
            u_jp1[:, N_x] = (
                -u_jm1[:, N_x]
                + 2 * u_j[:, N_x]
                + C2
                * (
                    0.5 * (q[N_x - 1] + q[N_x]) * (u_j[:, N_x - 1] - u_j[:, N_x])
                    - 0.5 * (q[N_x - 1] + q[N_x]) * (u_j[:, N_x] - u_j[:, N_x - 1])
                )
            )

        elif right_bound_cond == 3:
            # Mur bound cond
            # i = N_x
            u_jp1[:, N_x] = u_j[:, N_x - 1] + (CFL_2 - 1) / (CFL_2 + 1) * (
                u_jp1[:, N_x - 1] - u_j[:, N_x]
            )

        u_jm1[:, :] = u_j.clone()  # go to the next step
        u_j[:, :] = u_jp1.clone()  # go to the next step
        U[:, :, j] = u_j.clone()

    return U

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


# Define the initial condition function
def I(x):
    return np.exp(-((x - 0.5) ** 2) / 0.01)


# Define the velocity field function
def celer(x):
    # return np.sin(x*10)
    return torch.ones_like(x)  # constant velocity


# Set up the problem parameters
left_bound_cond = 2  # Mur boundary condition
right_bound_cond = 2  # Mur boundary condition
L_x = 1.5  # Range of the domain according to x
dx = 0.01  # Infinitesimal distance
L_t = 4  # Duration of simulation
# dt = 0.01 * dx  # Infinitesimal time
dt = dx
# Call the solve_wave_equation function
U = solve_wave_equation(10000, celer, left_bound_cond, right_bound_cond, L_x, dx, L_t, dt)

# Plot the results
X = np.linspace(0, L_x, U.shape[1])
T = np.linspace(0, L_t, U.shape[2])
plt.figure(figsize=(10, 6))
plt.imshow(U[0].T, extent=[0, L_x, 0, L_t], aspect="auto", cmap="viridis", origin="lower")
plt.colorbar(label="Amplitude")
plt.xlabel("Position (x)")
plt.ylabel("Time (t)")
plt.title("Solution of 1D Wave Equation")
plt.show()

In [ ]:
import viz_tools  # self-developed module that groups animation functions

anim1 = viz_tools.anim_1D(X, U[0], dt, 100, save=True, myxlim=(0, 1.5), myylim=(-2, 2))

In [ ]:
import torch.utils
import torch.utils.data

inputs = U.permute(0, 2, 1)[:, :-1, :]
outputs = U.permute(0, 2, 1)[:, 1:, :]
print(inputs.shape)
print(outputs.shape)
dataset = torch.utils.data.TensorDataset(inputs, outputs)

In [ ]:
file_path = "tensor.pt"
torch.save(dataset, file_path)

In [ ]:
import autoroot

from src.data.tensor_datamodule import TensorDataModule

d = TensorDataModule(path="./data/tensor.pt")

In [ ]:
d.dataset[0][0].shape